# Módulo 12 · Normalización, merge y exportación validada

Badge 3 · Pandas intermedio y análisis aplicado

## Normalización min-max: cambiar la escala, no los datos originales

### Interpretar la transformación

Resta el mínimo y divide entre máximo menos mínimo. El mínimo se transforma en 0 y el máximo en 1; se conservan las relaciones lineales entre valores. La nueva columna no tiene unidades monetarias y no elimina valores extremos. Conserva precio para interpretar y exportar.

### Resolver casos límite

Si todos los valores presentes son iguales, el denominador es cero. Puedes asignar cero a los valores presentes si documentas esa convención, conservando ausentes como ausentes. Si no hay valores válidos, no existe un mínimo útil: detén el cálculo con un mensaje. Verifica el rango de los valores presentes.

### Paso 1 · Prepara las herramientas

Importa las herramientas usadas en este ejemplo. Ejecuta este paso antes de continuar.

In [1]:
import pandas as pd

### Paso 2 · Construye los datos

Crea la tabla o la Series de práctica. Identifica sus valores, columnas e índice antes de transformarla.

In [2]:
precio = pd.Series([10.0, 15.0, 20.0, None])

### Paso 3 · Calcula y conserva el resultado

Lee la expresión de derecha a izquierda: primero se obtiene el resultado y luego se guarda en la variable o columna indicada. Una columna calculada debe recalcularse si cambian sus entradas.

In [3]:
lo, hi = (precio.min(), precio.max())

### Paso 4 · Resuelve el caso límite

Comprueba la condición antes de calcular. Sigue la rama que corresponda a los datos y observa cómo evita una operación inválida.

In [4]:
if pd.isna(lo) or pd.isna(hi):
    raise ValueError('No hay precios válidos')

### Paso 5 · Resuelve el caso límite

Comprueba la condición antes de calcular. Sigue la rama que corresponda a los datos y observa cómo evita una operación inválida.

In [5]:
if hi == lo:
    normalizado = precio.where(precio.isna(), 0.0)
else:
    normalizado = (precio - lo) / (hi - lo)

### Paso 6 · Observa la salida

Ejecuta esta instrucción y compara la salida con los valores que esperabas. Los nombres de columnas y el índice ayudan a ubicar cada resultado.

In [6]:
normalizado.tolist()

[0.0, 0.5, 1.0, nan]

### Paso 7 · Comprueba el resultado

Esta comprobación debe terminar sin error. Si falla, revisa los datos y el paso anterior antes de seguir.

In [7]:
assert normalizado.dropna().between(0, 1).all()

## Merge y exportación: integrar sin multiplicar ventas

### Elegir claves y tipo de unión

Cada venta contiene id_producto y el catálogo describe un producto por clave. inner conserva solo coincidencias; left conserva todas las ventas de la izquierda y permite detectar claves sin catálogo. La parte 3 pide inner y un catálogo que cubra todas las claves. Antes de unir, comprueba nulos, duplicados y compatibilidad de tipos.

### Validar la cardinalidad

many_to_one permite repetir la clave en ventas, pero exige unicidad en el catálogo. Si el catálogo contiene dos filas para una clave, una unión sin validar podría duplicar sus ventas e inflar ingresos. validate detecta esa cardinalidad incorrecta, pero no detecta por sí solo ventas descartadas por falta de coincidencia: comprueba también filas y totales.

### Exportar y volver a leer

index=False evita convertir el índice en una columna accidental. Relee el archivo y compara columnas, filas y valores con tolerancia decimal. CSV no conserva toda la información de tipos: fechas e identificadores con ceros iniciales pueden necesitar parámetros al leer. El ejemplo usa un CSV en memoria; el trabajo requiere archivos reales.

### Paso 1 · Prepara las herramientas

Importa las herramientas usadas en este ejemplo. Ejecuta este paso antes de continuar.

In [8]:
import pandas as pd
from io import StringIO

### Paso 2 · Construye los datos

Crea la tabla o la Series de práctica. Identifica sus valores, columnas e índice antes de transformarla.

In [9]:
ventas = pd.DataFrame({'id_producto': [1, 1, 2], 'importe': [20.0, 30.0, 40.0]})

### Paso 3 · Construye los datos

Crea la tabla o la Series de práctica. Identifica sus valores, columnas e índice antes de transformarla.

In [10]:
catalogo = pd.DataFrame({'id_producto': [1, 2], 'proveedor': ['Norte', 'Sur']})

### Paso 4 · Comprueba el resultado

Esta comprobación debe terminar sin error. Si falla, revisa los datos y el paso anterior antes de seguir.

In [11]:
assert catalogo['id_producto'].notna().all() and catalogo['id_producto'].is_unique

### Paso 5 · Integra las tablas

Relaciona las filas mediante la clave indicada. La validación exige la cardinalidad esperada; después comprueba también filas e importes.

In [12]:
integrado = ventas.merge(catalogo, on='id_producto', how='inner', validate='many_to_one')

### Paso 6 · Comprueba el resultado

Esta comprobación debe terminar sin error. Si falla, revisa los datos y el paso anterior antes de seguir.

In [13]:
assert len(integrado) == len(ventas)

### Paso 7 · Comprueba el resultado

Esta comprobación debe terminar sin error. Si falla, revisa los datos y el paso anterior antes de seguir.

In [14]:
assert abs(integrado['importe'].sum() - ventas['importe'].sum()) < 0.01

### Paso 8 · Calcula y conserva el resultado

Lee la expresión de derecha a izquierda: primero se obtiene el resultado y luego se guarda en la variable o columna indicada. Una columna calculada debe recalcularse si cambian sus entradas.

In [15]:
csv = StringIO()

### Paso 9 · Exporta la tabla

Guarda las columnas sin añadir el índice cuando se indica index=False. La verificación se completa al volver a leer el resultado.

In [16]:
integrado.to_csv(csv, index=False)

### Paso 10 · Observa la salida

Ejecuta esta instrucción y compara la salida con los valores que esperabas. Los nombres de columnas y el índice ayudan a ubicar cada resultado.

In [17]:
csv.seek(0)

0

### Paso 11 · Lee los datos

Carga el CSV en una tabla. Verifica a continuación su forma, nombres y tipos antes de interpretarlo.

In [18]:
leido = pd.read_csv(csv)

### Paso 12 · Comprueba el resultado

Esta comprobación debe terminar sin error. Si falla, revisa los datos y el paso anterior antes de seguir.

In [19]:
assert leido.columns.tolist() == integrado.columns.tolist()

### Paso 13 · Comprueba el resultado

Esta comprobación debe terminar sin error. Si falla, revisa los datos y el paso anterior antes de seguir.

In [20]:
assert len(leido) == len(integrado)

### Paso 14 · Comprueba el resultado

Esta comprobación debe terminar sin error. Si falla, revisa los datos y el paso anterior antes de seguir.

In [21]:
assert (leido['importe'] - integrado['importe']).abs().lt(0.01).all()

### Paso 15 · Observa la salida

Ejecuta esta instrucción y compara la salida con los valores que esperabas. Los nombres de columnas y el índice ayudan a ubicar cada resultado.

In [22]:
leido

,id_producto,importe,proveedor
0,1,20.0,Norte
1,1,30.0,Norte
2,2,40.0,Sur


## Práctica: Escala min-max

Normaliza [10,15,20]; explica el caso de precios constantes.

**Comprobación:** 0; 0,5; 1. Un rango constante requiere una regla que evite dividir entre cero.

In [ ]:
# Resuelve aquí.

## Práctica: Proteger una unión

Construye ventas con claves [1,1,2] y un catálogo con claves [1,2]. Une con many_to_one y comprueba filas.

**Comprobación:** Tres filas; las claves deben ser únicas en el catálogo.

In [ ]:
# Resuelve aquí.

## Trabajo grupal I

Parte 3 completa

- Leer ventas_limpias.csv de forma independiente.
- Completar la parte 3, incluyendo controles de rango, claves y filas.
- Releer datos_limpios.csv y comparar valores con tolerancia.

### Autoevaluación

¿Qué valida many_to_one?

1. Unicidad de claves en la tabla derecha
2. Igualdad de todos los valores
3. Ausencia de valores nulos en toda la tabla

**Respuesta y justificación:**

### Autoevaluación

¿Por qué releer el CSV?

1. Para generar más filas
2. Para comprobar estructura y valores tras exportar
3. Para recuperar automáticamente todos los tipos

**Respuesta y justificación:**